In [30]:
import os

import certifi

# Use certifi's trusted CA bundle for Python/torchvision downloads.
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt


**Note — Select a device**

PyTorch uses CUDA on supported NVIDIA GPUs, MPS on Apple silicon, and otherwise falls back to the CPU. The model and every input batch must use the same device.

In [31]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using:", device)

Using: mps


**Note — Prepare CIFAR-10**

CIFAR-10 contains 32 × 32 RGB images from ten classes. Random cropping and horizontal flipping create varied training examples, while normalization puts each color channel on a consistent scale. Test images are normalized without random augmentation.

In [32]:
# Use the reliable CIFAR-10 mirror used by the CNN tutorial.
datasets.CIFAR10.url = "https://huggingface.co/datasets/VerisimilitudeX/cifar10/resolve/main/cifar-10-python.tar.gz?download=true"

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

**Note — Load batches**

Each batch contains 128 images shaped `[128, 3, 32, 32]` and 128 class labels. Training data is shuffled each epoch; test data stays in a consistent order.

In [33]:
batch = 128

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch,
    shuffle=True,
    num_workers=2
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch,
    shuffle=False,
    num_workers=2
)

images, batches = next(iter(train_dataloader))
print(images.shape, batches.shape)


torch.Size([128, 3, 32, 32]) torch.Size([128])


**Note — The residual block**

A `BasicBlock` applies two 3 × 3 convolutions, then adds the original input through a shortcut connection. This gives gradients a direct path through the network. When shape or channel count changes, a 1 × 1 convolution adjusts the shortcut so the tensors can be added.

In [34]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Identity()

        if stride != 1 or in_channels != self.expansion * out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = nn.ReLU()(out)
        out = self.conv2(out)
        out = self.bn2(out)

        out += identity

        out = nn.ReLU()(out)
        return out

### Build ResNet-18

**Note — Stack residual stages**

The four stages use 64, 128, 256, and 512 channels. Stride 2 at the start of stages 2–4 halves the spatial dimensions. Adaptive average pooling reduces each final feature map to 1 × 1 before the ten-class linear layer.

In [35]:
class ResNet(nn.Module):

    def __init__(self, block, layers, num_classes=10):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(64)

        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(
            block,
            out_channels=64,
            blocks=layers[0],
            stride=1
        )

        self.layer2 = self._make_layer(
            block,
            out_channels=128,
            blocks=layers[1],
            stride=2
        )

        self.layer3 = self._make_layer(
            block,
            out_channels=256,
            blocks=layers[2],
            stride=2
        )

        self.layer4 = self._make_layer(
            block,
            out_channels=512,
            blocks=layers[3],
            stride=2
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Linear(
            512 * block.expansion,
            num_classes
        )

    def _make_layer(self, block, out_channels, blocks, stride):

        layers = []

        layers.append(
            block(
                self.in_channels,
                out_channels,
                stride
            )
        )

        self.in_channels = out_channels * block.expansion

        for _ in range(1, blocks):
            layers.append(
                block(
                    self.in_channels,
                    out_channels
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)

        x = self.layer2(x)

        x = self.layer3(x)

        x = self.layer4(x)

        x = self.avgpool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x

**Note — Why this is ResNet-18**

The configuration `[2, 2, 2, 2]` creates two basic residual blocks in each of four stages. Together with the first convolution and final linear layer, this forms the 18-layer ResNet architecture adapted for CIFAR-10.

In [36]:
def resnet18(num_classes=10):
    return ResNet(
        BasicBlock,
        [2, 2, 2, 2],
        num_classes=num_classes
    )

model = resnet18().to(device)

print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (shortcut): Identity()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bi

**Check — Output shape**

A test batch of 64 RGB images should produce `[64, 10]`: one raw logit for every CIFAR-10 class per image. This quick forward pass checks the architecture before training.

In [37]:
x = torch.randn(64, 3, 32, 32).to(device)

output = model(x)

print(output.shape)

torch.Size([64, 10])


**Note — Loss and optimization**

Cross-entropy compares the ten logits with the correct class. SGD updates the weights, momentum smooths the update direction, weight decay discourages overly large weights, and cosine annealing gradually lowers the learning rate.

In [38]:
criterion = nn.CrossEntropyLoss()


optimizer = optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100
)



**Note — Train one epoch**

For each batch, the function clears old gradients, performs a forward pass, calculates loss, backpropagates, and updates the parameters. The weighted loss and correct predictions are accumulated across the complete training set.

In [39]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


**Note — Evaluate without training**

`model.eval()` uses inference behavior for batch normalization, and `@torch.no_grad()` prevents gradient tracking. Evaluation measures performance without changing the learned weights.

In [40]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    loss = running_loss / total
    accuracy = 100 * correct / total

    return loss, accuracy

**Note — Run the experiment**

Each epoch trains once, evaluates once, advances the learning-rate schedule, and records the metrics. The current training function returns accuracy as a fraction from 0 to 1, while evaluation returns a percentage from 0 to 100; convert them to the same scale before directly comparing or plotting them.

In [ ]:
epochs = 50

train_losses = []
train_accuracies = []

test_losses = []
test_accuracies = []

for epoch in range(epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_dataloader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc = evaluate(
        model,
        test_dataloader,
        criterion,
        device
    )

    scheduler.step()

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Test Loss: {test_loss:.4f} "
        f"Test Acc: {test_acc:.2f}%"
    )

**Note — Read the curves and save the model**

Loss should generally decrease while accuracy increases. A widening gap between training and test performance can indicate overfitting. `state_dict()` saves the learned parameters so the same ResNet-18 architecture can load them later.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Train Loss")
plt.plot(test_losses, label="Test Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()


plt.figure(figsize=(8, 5))

plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(test_accuracies, label="Test Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()

plt.show()

torch.save(
    model.state_dict(),
    "resnet18_cifar10.pth"
)